We use content-based filtering recsys by computing similarity between movies.

In [1]:
import pandas as pd
import sqlite3

con = sqlite3.connect('../data/movies.db')
con.execute('PRAGMA foreign_keys = ON')  # enable foreign keys constraint and ON DELETE CASCADE

df = pd.read_sql_query("SELECT * FROM movie_detail", con)
df = df.drop('note', axis=1)
df.tail()

,id,name,year,status,type,country,genres,rating,watched_date
355,356,Saltburn,2023,waiting,movie,US,"thriller,dark comedy,psychological,drama",NaN,None
356,357,Bloody Flower,2026,waiting,series,Korea,"mystery,crime,thriller,psychological",NaN,None
357,358,Hoppers,2026,waiting,movie,US,"comedy,animation,sci-fi",NaN,None
358,359,Little Amélie or the Character of Rain,2025,waiting,movie,None,"animation,family,drama",NaN,None
359,360,Once We Were Us,2025,waiting,movie,Korea,"romance,drama",NaN,None


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            360 non-null    int64  
 1   name          360 non-null    object 
 2   year          360 non-null    int64  
 3   status        360 non-null    object 
 4   type          360 non-null    object 
 5   country       347 non-null    object 
 6   genres        360 non-null    object 
 7   rating        70 non-null     float64
 8   watched_date  70 non-null     object 
dtypes: float64(1), int64(2), object(6)
memory usage: 25.4+ KB


In [3]:
df.shape

(360, 9)

In [4]:
original_df = df.copy()

## Preprocessing

In [5]:
df = df.drop(columns=['rating', 'watched_date'])
df = df.reset_index(drop=True)
df.tail()

,id,name,year,status,type,country,genres
355,356,Saltburn,2023,waiting,movie,US,"thriller,dark comedy,psychological,drama"
356,357,Bloody Flower,2026,waiting,series,Korea,"mystery,crime,thriller,psychological"
357,358,Hoppers,2026,waiting,movie,US,"comedy,animation,sci-fi"
358,359,Little Amélie or the Character of Rain,2025,waiting,movie,None,"animation,family,drama"
359,360,Once We Were Us,2025,waiting,movie,Korea,"romance,drama"


## Feature similarity matrix

In [6]:
# Min-Max scaling for year, because: below-average year becomes a penalty -> not meaningful
# Min-Max scaling for year does not mean: the more recent the better
df_features = df.copy()
min_year, max_year =  df['year'].min(), df['year'].max()
df_features['year'] = (df['year'] - min_year) / (max_year - min_year)
df_features.tail()

,id,name,year,status,type,country,genres
355,356,Saltburn,0.956522,waiting,movie,US,"thriller,dark comedy,psychological,drama"
356,357,Bloody Flower,1.000000,waiting,series,Korea,"mystery,crime,thriller,psychological"
357,358,Hoppers,1.000000,waiting,movie,US,"comedy,animation,sci-fi"
358,359,Little Amélie or the Character of Rain,0.985507,waiting,movie,None,"animation,family,drama"
359,360,Once We Were Us,0.985507,waiting,movie,Korea,"romance,drama"


In [7]:
type_features = df['type'].str.get_dummies()
type_features.tail()

,movie,series
355,1,0
356,0,1
357,1,0
358,1,0
359,1,0


In [8]:
# df['country'] = df['country'].fillna('Unknown')
country_features = df['country'].str.get_dummies()
country_features.tail()

,China,Japan,Korea,US
355,0,0,0,1
356,0,0,1,0
357,0,0,0,1
358,0,0,0,0
359,0,0,1,0


In [9]:
genres_features = df['genres'].str.get_dummies(sep=',')
genres_features.tail()

,action,adventure,animation,biography,comedy,coming of age,crime,dark comedy,death,documentary,...,sci-fi,sitcom,sport,supernatural,survival,thriller,time travel,war,youth,zombie
355,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
356,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
357,0,0,1,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
358,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
359,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
feature_matrix = pd.concat(
    [
        df_features.drop(columns=['id', 'name', 'status', 'type', 'country', 'genres']), 
        type_features, country_features, genres_features
    ], axis=1
)
feature_matrix.tail()

,year,movie,series,China,Japan,Korea,US,action,adventure,animation,...,sci-fi,sitcom,sport,supernatural,survival,thriller,time travel,war,youth,zombie
355,0.956522,1,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
356,1.000000,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
357,1.000000,1,0,0,0,0,1,0,0,1,...,1,0,0,0,0,0,0,0,0,0
358,0.985507,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
359,0.985507,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
feature_matrix.shape

(360, 47)

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feature_matrix)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=df['id'],
    columns=df['id']
)
similarity_df.tail()

id,1,2,3,4,5,6,7,8,9,10,...,351,352,353,354,355,356,357,358,359,360
id,,,,,,,,,,,,,,,,,,,,,
356,0.250204,0.273922,0.154998,0.309617,0.153038,0.131837,0.156941,0.143061,0.171720,0.175846,...,0.135771,0.493781,0.148500,0.671043,0.668787,1.000000,0.424950,0.458999,0.501897,0.501897
357,0.252511,0.277318,0.332022,0.137730,0.330462,0.294527,0.333562,0.304061,0.370802,0.564768,...,0.284229,0.324014,0.308607,0.332022,0.325657,0.424950,1.000000,0.154303,0.167063,0.336582
358,0.450284,0.473220,0.173960,0.338855,0.171760,0.318126,0.176141,0.328423,0.192727,0.197359,...,0.461624,0.349975,0.500000,0.543290,0.725146,0.458999,0.154303,1.000000,0.546652,0.363550
359,0.103075,0.136263,0.188344,0.369901,0.185963,0.160201,0.190706,0.173839,0.208664,0.213678,...,0.334850,0.587551,0.363550,0.391219,0.383808,0.501897,0.167063,0.546652,1.000000,0.597685
360,0.103075,0.136263,0.391219,0.578735,0.389403,0.347141,0.393012,0.358253,0.208664,0.667031,...,0.164981,0.998862,0.180448,0.391219,0.383808,0.501897,0.336582,0.363550,0.597685,1.000000


In [13]:
similarity_df.shape

(360, 360)

## Movies recommendation

In [14]:
def recommend(movie_id, top_k=5):
    
    if movie_id not in similarity_df.index:
        return f"Movie ID {movie_id} not found."

    sim_scores = similarity_df.loc[movie_id].drop(movie_id)  # remove itself

    # Remove completed and dropped movies
    valid_ids = df[df['status'] == 'waiting']['id']
    sim_scores = sim_scores[sim_scores.index.isin(valid_ids)]
    
    top_movies = sim_scores.sort_values(ascending=False).head(top_k)    

    return df[df['id'].isin(top_movies.index)]

In [15]:
def get_movie(movie_id: int, cur: sqlite3.Cursor) -> tuple | None:
    """Get movie information by id. Return a tuple or None if not found."""
    cur.execute('SELECT * FROM movie_detail WHERE id = ?', (movie_id,))
    return cur.fetchone()

con.row_factory = sqlite3.Row  # for dictionary conversion
cur = con.cursor()

In [16]:
def recsys(movie_id, top_k=5):
    movie = dict(get_movie(movie_id, cur))
    movie.pop('note')
    print(movie)
    display(recommend(movie_id, top_k=top_k))

In [17]:
recsys(227)

{'id': 227, 'name': 'Go Back Couple', 'year': 2017, 'status': 'completed', 'type': 'series', 'country': 'Korea', 'genres': 'comedy,romance,time travel,life', 'rating': 10.0, 'watched_date': '2025-10-13'}


,id,name,year,status,type,country,genres
161,162,When Life Gives You Tangerines,2025,waiting,series,Korea,"romance,life"
181,182,Reply 1988,2015,waiting,series,Korea,"comedy,romance,life"
193,194,My Liberation Notes,2022,waiting,series,Korea,"romance,life"
230,231,Because This Is My First Life,2017,waiting,series,Korea,"comedy,romance,life"
326,327,Can This Love Be Translated?,2026,waiting,series,Korea,"comedy,romance"


In [18]:
recsys(300)

{'id': 300, 'name': 'Rango', 'year': 2011, 'status': 'completed', 'type': 'movie', 'country': 'US', 'genres': 'comedy,animation,adventure', 'rating': 8.0, 'watched_date': '2025-11-24'}


,id,name,year,status,type,country,genres
62,63,Soul,2020,waiting,movie,US,animation
94,95,Flow,2024,waiting,movie,US,"animation,adventure"
222,223,"The Boy, the Mole, the Fox and the Horse",2022,waiting,movie,US,"animation,family,adventure"
346,347,Ralph Breaks the Internet,2018,waiting,movie,US,"comedy,animation,adventure"
357,358,Hoppers,2026,waiting,movie,US,"comedy,animation,sci-fi"


In [19]:
recsys(340)

{'id': 340, 'name': 'Ne Zha (Na Tra)', 'year': 2019, 'status': 'completed', 'type': 'movie', 'country': 'China', 'genres': 'animation,family,adventure,fantasy', 'rating': 9.0, 'watched_date': '2026-01-25'}


,id,name,year,status,type,country,genres
70,71,Deep Sea,2023,waiting,movie,China,"animation,adventure,fantasy"
154,155,The Legend of Hei,2019,waiting,movie,China,"animation,action,adventure,fantasy"
233,234,The Legend of Hei 2,2025,waiting,movie,China,"animation,action,adventure,fantasy"
304,305,Nobody,2025,waiting,movie,China,"comedy,animation,adventure,fantasy"
340,341,Ne Zha 2 (Na Tra 2),2025,waiting,movie,China,"animation,action,adventure,fantasy"


In [20]:
recsys(248)

{'id': 248, 'name': 'Mickey 17', 'year': 2025, 'status': 'completed', 'type': 'movie', 'country': 'US', 'genres': 'adventure,sci-fi,dark comedy', 'rating': 8.0, 'watched_date': '2025-12-14'}


,id,name,year,status,type,country,genres
55,56,Dune,2021,waiting,movie,US,"action,adventure,sci-fi"
80,81,The Martian,2015,waiting,movie,US,"adventure,sci-fi"
110,111,Companion,2025,waiting,movie,US,"sci-fi,thriller,dark comedy"
115,116,Poor Things,2023,waiting,movie,US,"romance,sci-fi,dark comedy"
342,343,Project Hail Mary,2026,waiting,movie,US,"adventure,sci-fi,mystery"


In [21]:
recsys(358)

{'id': 358, 'name': 'Hoppers', 'year': 2026, 'status': 'waiting', 'type': 'movie', 'country': 'US', 'genres': 'comedy,animation,sci-fi', 'rating': None, 'watched_date': None}


,id,name,year,status,type,country,genres
61,62,Her,2013,waiting,movie,US,"comedy,romance,sci-fi"
62,63,Soul,2020,waiting,movie,US,animation
283,284,Arco,2025,waiting,movie,US,"animation,sci-fi,fantasy"
300,301,Robot Dreams,2023,waiting,movie,None,"comedy,animation,family,sci-fi"
346,347,Ralph Breaks the Internet,2018,waiting,movie,US,"comedy,animation,adventure"


## User profile

In [22]:
df = original_df
df.tail()

,id,name,year,status,type,country,genres,rating,watched_date
355,356,Saltburn,2023,waiting,movie,US,"thriller,dark comedy,psychological,drama",NaN,None
356,357,Bloody Flower,2026,waiting,series,Korea,"mystery,crime,thriller,psychological",NaN,None
357,358,Hoppers,2026,waiting,movie,US,"comedy,animation,sci-fi",NaN,None
358,359,Little Amélie or the Character of Rain,2025,waiting,movie,None,"animation,family,drama",NaN,None
359,360,Once We Were Us,2025,waiting,movie,Korea,"romance,drama",NaN,None


In [23]:
watched_df = df[df['status'] == 'completed']
unwatched_ids = df[df['status'] == 'waiting']['id']

### Recent user profile

In [24]:
recent = watched_df.sort_values('watched_date', ascending=False).head(5)
recent

,id,name,year,status,type,country,genres,rating,watched_date
313,314,Marty Supreme,2025,completed,movie,US,"sport,drama",9.0,2026-02-27
350,351,The Daily Life of the Immortal King 5,2025,completed,series,China,"comedy,animation,action,adventure",7.0,2026-02-22
339,340,Ne Zha (Na Tra),2019,completed,movie,China,"animation,family,adventure,fantasy",9.0,2026-01-25
165,166,Sing 2,2021,completed,movie,US,"animation,music",7.0,2026-01-20
228,229,KPop Demon Hunters,2025,completed,movie,US,"animation,action,music,fantasy",9.0,2025-12-25


In [25]:
print(recent.index)
feature_matrix.loc[recent.index]

Index([313, 350, 339, 165, 228], dtype='int64')


,year,movie,series,China,Japan,Korea,US,action,adventure,animation,...,sci-fi,sitcom,sport,supernatural,survival,thriller,time travel,war,youth,zombie
313,0.985507,1,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
350,0.985507,0,1,1,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
339,0.898551,1,0,1,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
165,0.927536,1,0,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
228,0.985507,1,0,0,0,0,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0


In [26]:
user_profile = feature_matrix.loc[recent.index].mean(axis=0)
similarity_scores = cosine_similarity([user_profile], feature_matrix).flatten()
similarity_scores.shape

(360,)

In [27]:
sim_series = pd.Series(similarity_scores, index=df['id'])
sim_series = sim_series[sim_series.index.isin(unwatched_ids)]
top_ids = sim_series.sort_values(ascending=False).head(10).index

# preserve ranking orders
df.set_index('id').loc[top_ids].reset_index().drop(columns=['rating', 'watched_date'])

,id,name,year,status,type,country,genres
0,95,Flow,2024,waiting,movie,US,"animation,adventure"
1,63,Soul,2020,waiting,movie,US,animation
2,341,Ne Zha 2 (Na Tra 2),2025,waiting,movie,China,"animation,action,adventure,fantasy"
3,234,The Legend of Hei 2,2025,waiting,movie,China,"animation,action,adventure,fantasy"
4,155,The Legend of Hei,2019,waiting,movie,China,"animation,action,adventure,fantasy"
5,71,Deep Sea,2023,waiting,movie,China,"animation,adventure,fantasy"
6,223,"The Boy, the Mole, the Fox and the Horse",2022,waiting,movie,US,"animation,family,adventure"
7,347,Ralph Breaks the Internet,2018,waiting,movie,US,"comedy,animation,adventure"
8,323,Into The Mortal World,2024,waiting,movie,China,"animation,fantasy"
9,346,GOAT,2026,waiting,movie,US,"comedy,animation,action,sport"


#### Rating weighting

In [28]:
def recommend_user_profile(user_profile, top_k: int = 10) -> pd.DataFrame:
    """Recommend based on provided user profile."""
    similarity_scores = cosine_similarity([user_profile], feature_matrix).flatten()
    sim_series = pd.Series(similarity_scores, index=df['id'])
    sim_series = sim_series[sim_series.index.isin(unwatched_ids)]
    top_ids = sim_series.sort_values(ascending=False).head(top_k).index
    return df.set_index('id').loc[top_ids].reset_index().drop(columns=['rating', 'watched_date'])

In [29]:
# scale rating
rating_weights = recent['rating'] / 10
rating_weights

313    0.9
350    0.7
339    0.9
165    0.7
228    0.9
Name: rating, dtype: float64

In [30]:
import numpy as np

watched_features = feature_matrix.loc[recent.index]
user_profile_rating_weighted = np.average(watched_features, axis=0, weights=rating_weights)

In [31]:
recent

,id,name,year,status,type,country,genres,rating,watched_date
313,314,Marty Supreme,2025,completed,movie,US,"sport,drama",9.0,2026-02-27
350,351,The Daily Life of the Immortal King 5,2025,completed,series,China,"comedy,animation,action,adventure",7.0,2026-02-22
339,340,Ne Zha (Na Tra),2019,completed,movie,China,"animation,family,adventure,fantasy",9.0,2026-01-25
165,166,Sing 2,2021,completed,movie,US,"animation,music",7.0,2026-01-20
228,229,KPop Demon Hunters,2025,completed,movie,US,"animation,action,music,fantasy",9.0,2025-12-25


- highly rated movies have **sport**, **animation**, and **fantasy** genre (9), while **comedy** genre has lower rating (7). 
- movies are from US and China

In [32]:
recommend_user_profile(user_profile_rating_weighted)

,id,name,year,status,type,country,genres
0,95,Flow,2024,waiting,movie,US,"animation,adventure"
1,63,Soul,2020,waiting,movie,US,animation
2,234,The Legend of Hei 2,2025,waiting,movie,China,"animation,action,adventure,fantasy"
3,341,Ne Zha 2 (Na Tra 2),2025,waiting,movie,China,"animation,action,adventure,fantasy"
4,155,The Legend of Hei,2019,waiting,movie,China,"animation,action,adventure,fantasy"
5,71,Deep Sea,2023,waiting,movie,China,"animation,adventure,fantasy"
6,223,"The Boy, the Mole, the Fox and the Horse",2022,waiting,movie,US,"animation,family,adventure"
7,151,Coraline,2009,waiting,movie,US,"animation,family,fantasy"
8,323,Into The Mortal World,2024,waiting,movie,China,"animation,fantasy"
9,347,Ralph Breaks the Internet,2018,waiting,movie,US,"comedy,animation,adventure"


In [33]:
# Recommended result when not using rating-weighted:
# ...
# 6	223	The Boy, the Mole, the Fox and the Horse	2022	waiting	movie	US	animation,family,adventure	NaN	None
# 7	347	Ralph Breaks the Internet	2018	waiting	movie	US	comedy,animation,adventure	NaN	None
# 8	323	Into The Mortal World	2024	waiting	movie	China	animation,fantasy	NaN	None
# 9	346	GOAT	2026	waiting	movie	US	comedy,animation,action,sport	NaN	None

The recommended result when using rating-weighted:
- move 'Ralph Breaks the Internet' to last because it has comedy genres, whereas if we don't use rating-weighted, 'Ralph Breaks the Internet' was ranked before 'Into The Mortal World', which have 2 high-rated genres: animation and fantasy
- recommend 'Coraline' which has animation and fantasy genres (high-rated), whereas not using rating-weighted doesn't

#### Nonlinear rating weighting

In [34]:
nonlinear_rating_weights = (recent['rating'] / 10)**2
nonlinear_rating_weights

313    0.81
350    0.49
339    0.81
165    0.49
228    0.81
Name: rating, dtype: float64

In [35]:
recommend_user_profile(
    np.average(watched_features, axis=0, weights=nonlinear_rating_weights)
)

,id,name,year,status,type,country,genres
0,95,Flow,2024,waiting,movie,US,"animation,adventure"
1,63,Soul,2020,waiting,movie,US,animation
2,341,Ne Zha 2 (Na Tra 2),2025,waiting,movie,China,"animation,action,adventure,fantasy"
3,234,The Legend of Hei 2,2025,waiting,movie,China,"animation,action,adventure,fantasy"
4,155,The Legend of Hei,2019,waiting,movie,China,"animation,action,adventure,fantasy"
5,151,Coraline,2009,waiting,movie,US,"animation,family,fantasy"
6,71,Deep Sea,2023,waiting,movie,China,"animation,adventure,fantasy"
7,223,"The Boy, the Mole, the Fox and the Horse",2022,waiting,movie,US,"animation,family,adventure"
8,323,Into The Mortal World,2024,waiting,movie,China,"animation,fantasy"
9,347,Ralph Breaks the Internet,2018,waiting,movie,US,"comedy,animation,adventure"


- 'Coraline' ranked above 'Deep Sea' because adventure genre from 'Deep Sea' has higher penalty (7 rating -> 0.49) compare to family genre of 'Coraline'

### All user profile

In [36]:
df = original_df
df.tail()
watched_df = df[df['status'] == 'completed']
unwatched_ids = df[df['status'] == 'waiting']['id']
watched_df.shape

(66, 9)

In [37]:
def build_item_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """Build a numeric feature matrix for item-based recommendation."""

    df_features = df.copy()
    df_features = df_features.drop(columns=['rating', 'watched_date'])

    # Min-max scaling year column
    min_year, max_year =  df['year'].min(), df['year'].max()
    df_features['year'] = (df['year'] - min_year) / (max_year - min_year)
    df_features['year'] *= 0.2  # Downweight because same year doesn't mean high similarity

    country_features = df['country'].str.get_dummies()
    country_features *= 0.4
    
    type_features = df['type'].str.get_dummies()
    type_features *= 0.5
    
    genres_features = df['genres'].str.get_dummies(',')

    return pd.concat(
        [
            df_features.drop(columns=['id', 'name', 'status', 'type', 'country', 'genres']), 
            type_features, country_features, genres_features
        ], axis=1
    )

# Build item feature matrix with weights for year, country, and type
feature_matrix = build_item_feature_matrix(df)

In [38]:
watched_features = feature_matrix.loc[watched_df.index]
watched_features.shape

(66, 47)

In [39]:
# redefine for all watched movies
def recommend_user_profile(user_profile, top_k: int = 10) -> pd.DataFrame:
    """Recommend based on provided user profile."""
    similarity_scores = cosine_similarity([user_profile], feature_matrix).flatten()
    sim_series = pd.Series(similarity_scores, index=df['id'])
    sim_series = sim_series.loc[unwatched_ids]
    top_ids = sim_series.sort_values(ascending=False).head(top_k).index
    return df.set_index('id').loc[top_ids].reset_index().drop(columns=['rating', 'watched_date'])

In [40]:
user_profile = np.average(watched_features, axis=0, weights=None)
recommend_user_profile(user_profile)

,id,name,year,status,type,country,genres
0,262,Scissor Seven,2018,waiting,series,China,"comedy,romance,animation,action,adventure"
1,327,Can This Love Be Translated?,2026,waiting,series,Korea,"comedy,romance"
2,188,Love Untangled,2025,waiting,movie,Korea,"comedy,romance,school"
3,195,Goblin,2016,waiting,series,Korea,"comedy,romance,fantasy"
4,332,The Drama,2026,waiting,movie,US,"comedy,romance"
5,331,People We Meet on Vacation,2026,waiting,movie,US,"comedy,romance"
6,168,Materialists,2025,waiting,movie,US,"comedy,romance"
7,326,The Threesome,2025,waiting,movie,US,"comedy,romance"
8,67,Anora,2024,waiting,movie,US,"comedy,romance"
9,199,No Hard Feelings,2023,waiting,movie,US,"comedy,romance"


-> ~~Rom-Com god 👑~~ Watched lot of rom-com long time ago

#### Time decay weighting

In [41]:
def half_life(days) -> float:
    """Convert a half-life value (in days) to the exponential decay rate."""
    return np.log(2) / days

def exponential_decay(t, decay_rate: float = 0.01) -> np.ndarray:
    """Compute exponential decay weights."""
    return np.exp(-decay_rate*t)

def normalize_watched_date(date_str):
    """Normalize watched_date values into pandas Timestamp."""
    if pd.isna(date_str):
        return None
    
    date_str = str(date_str)

    # convert to mid-year for year-only and year with month data, mid-year is 1 July
    if len(date_str) == 4:  # YYYY
        return pd.Timestamp(f'{date_str}-07-01')

    if len(date_str) == 7:  # YYYY-MM
        return pd.Timestamp(f'{date_str}-15')

    return pd.Timestamp(date_str)

In [42]:
watched_date = watched_df['watched_date'].apply(normalize_watched_date)
today = pd.Timestamp.today().normalize()  # .normalize() removes the time portion
days_since_watch = (today - watched_date).dt.days

# more about decay rate in notebooks/time_decay.ipynb
time_decay_weights = exponential_decay(days_since_watch, decay_rate=half_life(180))

user_profile = np.average(watched_features, axis=0, weights=time_decay_weights)
recommend_user_profile(user_profile)

,id,name,year,status,type,country,genres
0,262,Scissor Seven,2018,waiting,series,China,"comedy,romance,animation,action,adventure"
1,305,Nobody,2025,waiting,movie,China,"comedy,animation,adventure,fantasy"
2,347,Ralph Breaks the Internet,2018,waiting,movie,US,"comedy,animation,adventure"
3,335,Chainsaw Man – The Movie: Reze Arc,2025,waiting,movie,Japan,"romance,animation,action,fantasy"
4,341,Ne Zha 2 (Na Tra 2),2025,waiting,movie,China,"animation,action,adventure,fantasy"
5,234,The Legend of Hei 2,2025,waiting,movie,China,"animation,action,adventure,fantasy"
6,155,The Legend of Hei,2019,waiting,movie,China,"animation,action,adventure,fantasy"
7,225,Everything Everywhere All at Once,2022,waiting,movie,US,"comedy,action,adventure,sci-fi"
8,157,Fullmetal Alchemist: Brotherhood,2009,waiting,series,Japan,"animation,action,adventure,fantasy"
9,238,Ponyo,2008,waiting,movie,Japan,"comedy,animation,family,adventure"


-> Less rom-com but still a lot

#### Nonlinear rating + Time decay weighting

In [43]:
rating_weights = (watched_df['rating'] / 10) ** 2  # see notebooks/rating_weight.ipynb
weights = rating_weights * time_decay_weights
user_profile = np.average(watched_features, axis=0, weights=weights)
recommend_user_profile(user_profile)

,id,name,year,status,type,country,genres
0,262,Scissor Seven,2018,waiting,series,China,"comedy,romance,animation,action,adventure"
1,335,Chainsaw Man – The Movie: Reze Arc,2025,waiting,movie,Japan,"romance,animation,action,fantasy"
2,225,Everything Everywhere All at Once,2022,waiting,movie,US,"comedy,action,adventure,sci-fi"
3,305,Nobody,2025,waiting,movie,China,"comedy,animation,adventure,fantasy"
4,321,Eternity,2025,waiting,movie,US,"comedy,romance,fantasy"
5,341,Ne Zha 2 (Na Tra 2),2025,waiting,movie,China,"animation,action,adventure,fantasy"
6,234,The Legend of Hei 2,2025,waiting,movie,China,"animation,action,adventure,fantasy"
7,155,The Legend of Hei,2019,waiting,movie,China,"animation,action,adventure,fantasy"
8,195,Goblin,2016,waiting,series,Korea,"comedy,romance,fantasy"
9,347,Ralph Breaks the Internet,2018,waiting,movie,US,"comedy,animation,adventure"


-> Perfect ❤️

## Hybrid Recommender Sytem

- [ ] Dimension reduction if it slow (SVD, PCA)

## Clean up

In [44]:
con.close()